# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

In [ ]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
import json
import os
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

# 1. Đọc dữ liệu từ data/splits/train.jsonl
train_path = "data/splits/train.jsonl"
with open(train_path, "r", encoding="utf-8") as f:
    train_data = [json.loads(line) for line in f]

total_images = len(train_data)
total_boxes = sum(len(item["objects"]) for item in train_data)
print(f"Tổng số ảnh tập train: {total_images}")
print(f"Tổng số bounding boxes tập train: {total_boxes}")

# 2. Trực quan hóa 6 ảnh mẫu
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx in range(min(6, total_images)):
    item = train_data[idx]
    img_path = os.path.join("data", item["image_path"])
    img = Image.open(img_path)
    w_img, h_img = item["width"], item["height"]

    ax = axes[idx]
    ax.imshow(img)
    ax.set_title(f"ID: {item['image_id'][:10]}...", fontsize=10)
    ax.axis("off")

    # Vẽ từng bbox
    for obj in item["objects"]:
        label = obj["label"]
        xc, yc, w, h = obj["yolo"]
        
        # Quy đổi tọa độ YOLO [0, 1] sang Toạ độ Pixel [x1, y1, width, height]
        x1 = (xc - w / 2) * w_img
        y1 = (yc - h / 2) * h_img
        bw = w * w_img
        bh = h * h_img

        rect = patches.Rectangle((x1, y1), bw, bh, linewidth=2, edgecolor="red", facecolor="none")
        ax.add_patch(rect)
        ax.text(x1, max(0, y1 - 5), label, color="white", fontsize=9, 
                weight="bold", bbox=dict(facecolor="red", alpha=0.7, pad=1, edgecolor="none"))

plt.tight_layout()
plt.show()

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
# YOUR CODE HERE: viết yolo_to_box() và objects_to_answer() (GUIDE bước 2)

## 3. E0: zero-shot

In [ ]:
# YOUR CODE HERE: nạp model 4-bit, suy luận trên test → work/e0.jsonl (GUIDE bước 3)

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# YOUR CODE HERE: encode_example() và collate() (GUIDE bước 4)

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# YOUR CODE HERE: apply_setup(model, setup) (GUIDE bước 5)

## 6. Vòng huấn luyện

In [ ]:
# YOUR CODE HERE: vòng train, lưu adapter + run.json (GUIDE bước 6)

## 7. Suy luận và chấm E1–E3

In [ ]:
# YOUR CODE HERE: suy luận test → work/eN.jsonl rồi chạy scripts/score.py với --meta (GUIDE bước 7)

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# YOUR CODE HERE: GUIDE bước 8 và 9